# 제3 테스트셋 (BFCL) 생성 (논문에 없는 추가)
BFCL v4 질의에 우리 함수 기준 정답을 붙인 test 셋(설명서 `BFCL_TEST.md`)으로 우리 모델과 공개 모델의 출력을 만든다.
생성·채점 모두 `src/evaluate.py` (BFCL 필드 `score_args`·`args_allowed` 처리 포함), 확률은 `src/probs.py`.

- 런타임: **A100 GPU**. 🔑 Secrets에 `HF_TOKEN`
- test 파일: Drive `MyDrive/octopus_v2/bfcl_test_draft.jsonl` (234개, **사용자 검토 전 초안**). 검토 후 바뀐 줄만 다시 채점하면 된다
- 결과: Drive `MyDrive/octopus_v2/runs/eval_bfcl/{ours,public}/` 에 `preds.jsonl`, `probs.jsonl`

## 준비 (그냥 실행)

In [1]:
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
Mounted at /content/drive
Cloning into '/content/octopus_v2_reproduce'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 224 (delta 120), reused 142 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (224/224), 2.14 MiB | 25.77 MiB/s, done.
Resolving deltas: 100% (120/120), done.
/content/octopus_v2_reproduce
Already up to date.
bb6f043 (HEAD -> main, origin/main,

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print("transformers", transformers.__version__)  # 4.56.2 여야 함

RUN_DIR = "/content/drive/MyDrive/octopus_v2/runs/full"   # 4단계에서 학습한 모델
EVAL = "/content/drive/MyDrive/octopus_v2/runs/eval_bfcl"
TEST = "/content/drive/MyDrive/octopus_v2/bfcl_test_draft.jsonl"
os.makedirs(EVAL, exist_ok=True)
!wc -l {TEST}

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
transformers 4.56.2
234 /content/drive/MyDrive/octopus_v2/bfcl_test_draft.jsonl


---
## 7-1. 우리 모델로 생성 + 확률
**예측**: 우리 모델은 질문형 negative 로 학습해서 질문형 positive(62개)를 거절할 것. 거절 정확도는 높을 것.
> (여기에 예측)

In [3]:
!python -m src.evaluate generate --model {RUN_DIR} --test-file {TEST} --out {EVAL}/ours/preds.jsonl
!python -m src.probs generate --model {RUN_DIR} --test-file {TEST} --preds {EVAL}/ours/preds.jsonl --out {EVAL}/ours/probs.jsonl

[생성] 234개 중 이미 234개 있음 → 0개 생성 → /content/drive/MyDrive/octopus_v2/runs/eval_bfcl/ours/preds.jsonl
[확률] 234개 중 이미 0개 있음 → 234개 생성 → /content/drive/MyDrive/octopus_v2/runs/eval_bfcl/ours/probs.jsonl
I0000 00:00:1794972937.400618    1456 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794972937.481221    1456 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794972939.831786    1456 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders.

In [4]:
!python -m src.evaluate score --preds {EVAL}/ours/preds.jsonl --test-file {TEST}

                                                      전체   seen_in_train=false
개수                                                   234                   234
형식 OK                                            99.57%                99.57% 
함수 선택 정확도                                        75.21%                75.21% 
  positive 함수 선택                                 57.26%                57.26% 
인자 정확도 (함수 맞힌 positive 중)                        84.78%                84.78% 
전체 정확도 ★                                         70.09%                70.09% 
  positive                                       51.28%                51.28% 
  negative                                       88.89%                88.89% 
느슨한 정확도 (논문에 없는 추가)                              70.51%                70.51% 
호출 문자열 완전 일치                                     61.11%                61.11% 

[결과 종류별 개수]
  args:count_mismatch                                    9                     9
  args:rule_violation                

---
## 7-2. 공개 모델 `NexaAIDev/Octopus-v2` 로 생성 + 확률
**예측**: 잡담·상식·계산 negative 에 함수를 부를 것 (우리 test 에서 negative 39%).
> (여기에 예측)

In [5]:
!python -m src.evaluate generate --model NexaAIDev/Octopus-v2 --test-file {TEST} --out {EVAL}/public/preds.jsonl
!python -m src.probs generate --model NexaAIDev/Octopus-v2 --test-file {TEST} --preds {EVAL}/public/preds.jsonl --out {EVAL}/public/probs.jsonl

[생성] 234개 중 이미 0개 있음 → 234개 생성 → /content/drive/MyDrive/octopus_v2/runs/eval_bfcl/public/preds.jsonl
tokenizer_config.json: 6.35kB [00:00, 1.90MB/s]
tokenizer.model: 100% 4.24M/4.24M [00:05<00:00, 735kB/s]
tokenizer.json: 100% 17.5M/17.5M [00:04<00:00, 4.35MB/s]
added_tokens.json: 100% 500/500 [00:00<00:00, 4.24MB/s]
special_tokens_map.json: 100% 957/957 [00:00<00:00, 8.24MB/s]
config.json: 100% 673/673 [00:00<00:00, 4.62MB/s]
I0000 00:00:1794973123.755841    2316 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794973123.828201    2316 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the a

In [6]:
!python -m src.evaluate score --preds {EVAL}/public/preds.jsonl --test-file {TEST}

                                                      전체   seen_in_train=false
개수                                                   234                   234
형식 OK                                            96.58%                96.58% 
함수 선택 정확도                                        62.82%                62.82% 
  positive 함수 선택                                 89.74%                89.74% 
인자 정확도 (함수 맞힌 positive 중)                        74.14%                74.14% 
전체 정확도 ★                                         56.41%                56.41% 
  positive                                       76.92%                76.92% 
  negative                                       35.90%                35.90% 
느슨한 정확도 (논문에 없는 추가)                              56.41%                56.41% 
호출 문자열 완전 일치                                     35.04%                35.04% 

[결과 종류별 개수]
  args:count_mismatch                                    3                     3
  args:rule_violation                

---
## 끝나면
결과는 Drive `runs/eval_bfcl/` 에 있다. 랩탑에서 Claude 에게 "BFCL 결과 분석해줘" 라고 하면 Drive 에서 받아 AUROC·오답 분석을 한다.